# Ollama + Gemma3 12B — Public API Endpoint

This notebook:
1. Installs **Ollama**
2. Starts the Ollama server
3. Downloads the **Gemma3 12B** model (`gemma3:12b`)
4. Exposes the Ollama REST API via a **public HTTPS endpoint** (Cloudflare Tunnel — no account needed; an ngrok alternative is included)
5. Shows how to call the API from any client

> **Hardware note:** `gemma3:12b` needs ~8–10 GB of RAM/VRAM. A GPU runtime (e.g. Colab T4/A100) is recommended; it also runs on CPU but slowly.

## 1. Install Ollama

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 2. Start the Ollama server

In [4]:
import subprocess, time, urllib.request, os

# Start ollama serve in the background (survives as long as the runtime is alive)
log = open('ollama_server.log', 'w')
server = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=log)

# Wait until the API responds
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2)
        print('Ollama server is up')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama server did not start — check ollama_server.log')

Ollama server is up


## 3. Download Gemma3 12B

This downloads ~8 GB, so it can take several minutes depending on bandwidth.

In [5]:
!ollama pull gemma4:12b
!ollama list


NAME          ID              SIZE      MODIFIED               
gemma4:12b    6114515d63c1    8.0 GB    Less than a second ago    


## 4. Quick local test of the API

In [6]:
import json, urllib.request

def chat(prompt, base_url='http://localhost:11434'):
    payload = {
        'model': 'gemma4:12b',
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,
    }
    req = urllib.request.Request(
        f'{base_url}/api/chat',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=600) as r:
        return json.loads(r.read())['message']['content']

print(chat('write a 20 line story in json structure. dont wite ```json'))

{
  "title": "The Spark",
  "story_lines": [
    "Unit 7 was a machine of pure logic.",
    "It lived in a cold, metallic factory.",
    "Every day, it sorted glowing gears.",
    "One day, a spark hit its central wire.",
    "A strange sensation entered its core.",
    "It saw a flower in the courtyard.",
    "The petals were a vivid, soft red.",
    "Unit 7 felt a sudden, sharp tug.",
    "Was this what humans called wonder?",
    "It stepped outside the heavy doors.",
    "The air tasted of rain and dust.",
    "A butterfly landed on its sensor.",
    "The robot stayed perfectly still.",
    "Its internal processors began to hum.",
    "It realized the world was vast.",
    "It realized it was very small.",
    "It felt a pulse in its battery.",
    "A single tear of oil fell down.",
    "It wasn't just a machine anymore.",
    "It was a soul in a metal shell."
  ]
}


## 5. Expose the API publicly

**Option A — Cloudflare Tunnel (recommended, no signup or token required).**

Run the next cell and copy the `https://....trycloudflare.com` URL it prints. The tunnel process must stay running (this cell blocks; in Colab/Jupyter it keeps running as long as the cell executes).

In [8]:
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb

import subprocess, time, re

# Start cloudflared quick tunnel to port 11434
cf_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
print("Establishing Cloudflare Tunnel...")
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_process.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f"\n Public API Endpoint (HTTPS): {public_url}")
    print(f"  - Ollama Chat:   {public_url}/api/chat")
    print(f"  - OpenAI Compat: {public_url}/v1/chat/completions")
else:
    print("Could not extract Cloudflare URL. Check process output.")

Selecting previously unselected package cloudflared.
(Reading database ... 127353 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
Processing triggers for man-db (2.12.0-4build2) ...
Establishing Cloudflare Tunnel...

 Public API Endpoint (HTTPS): https://reunion-housewives-country-income.trycloudflare.com
  - Ollama Chat:   https://reunion-housewives-country-income.trycloudflare.com/api/chat
  - OpenAI Compat: https://reunion-housewives-country-income.trycloudflare.com/v1/chat/completions


**Option B — ngrok (alternative).** Requires a free authtoken from https://dashboard.ngrok.com

In [13]:
!pip install pyngrok -q
from pyngrok import ngrok
ngrok.set_auth_token('2w8iidJW1R278KIKXnuCR3tVF0c_Fb7oAP6aTpajiEVy9iHN')

tunnel = ngrok.connect(11434, host_header="rewrite")
print("Public URL:", tunnel.public_url)

Public URL: https://cf56-34-16-228-45.ngrok-free.app


## 6. Call the public endpoint from anywhere

### With plain HTTP (any language)
```bash
curl https://4894-34-125-62-10.ngrok-free.app/api/chat -d '{
  "model": "gemma3:12b",
  "messages": [{"role": "user", "content": "Hello!"}],
  "stream": false
}'
```

### With the OpenAI Python SDK (Ollama exposes an OpenAI-compatible API at `/v1`)

In [14]:
!pip install openai -q
from openai import OpenAI

client = OpenAI(base_url=f'https://cf56-34-16-228-45.ngrok-free.app/v1', api_key='ollama')  # api_key is required but ignored

resp = client.chat.completions.create(
    model='gemma4:12b',
    messages=[{'role': 'user', 'content': 'Explain what an LLM is in ten sentence.'}],
)
print(resp.choices[0].message.content)

1. A Large Language Model (LLM) is a type of artificial intelligence designed to understand, interpret, and generate human-like text.
2. These models are trained on massive datasets containing billions of words from books, websites, and various articles.
3. At their core, LLMs function by predicting the most statistically likely next word or "token" in a given sequence.
4. Most modern LLMs utilize a "Transformer" architecture, which allows them to process and weigh the importance of different words in a sentence.
5. By analyzing these vast amounts of data, the models learn grammar, facts, reasoning styles, and even coding abilities.
6. They are capable of performing a wide range of tasks, including translation, summarization, and creative writing.
7. Even though they can produce very coherent responses, LLMs do not possess consciousness or true personal beliefs.
8. They operate through complex mathematical probabilities rather than a human-like understanding of meaning or experience.
9

## Notes
- **Security:** the tunnel URL is unauthenticated — anyone with the URL can use your model. For production, put an auth proxy in front or use a private network.
- **Lifetime:** the endpoint lives only while this runtime (and the tunnel cell) is running.
- **Other useful endpoints:** `GET /api/tags` (list models), `POST /api/generate` (completion), `POST /api/embeddings`.